<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.5_Praktikum_Overfitting_dan_Underfitting.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 5.5: Praktikum Komputasi - Overfitting dan Underfitting dalam Machine Learning
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Tujuan Praktikum:
1. Memvisualisasikan fenomena *Underfitting* dan *Overfitting* melalui variasi kapasitas hipotesis polinomial.
2. Membuktikan dekomposisi matematis $\text{Expected MSE} = \text{Bias}^2 + \text{Variance} + \sigma^2$ secara numerik menggunakan simulasi *bootstrap*.
3. Menganalisis kurva pembelajaran (*learning curves*) untuk mendeteksi *generalization gap*.
4. Menerapkan teknik regularisasi matematis (Ridge dan Lasso) untuk mengontrol variansi model.
5. Mengimplementasikan pemangkasan pohon keputusan (*Cost-Complexity Pruning* / `ccp_alpha`) untuk mencegah penghafalan derau data.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
print("Environment komputasi praktikum berhasil diinisialisasi.")


## 1. Eksperimen Kapasitas Model Polinomial: Respon Dosis Pupuk Nitrogen

Mari kita simulasikan hubungan non-linier antara dosis pupuk nitrogen ($kg/ha$) dan estimasi biomassa kelapa sawit ($ton/ha$).
Fungsi sejati: $f(x) = 12 + 3.2 x - 0.22 x^2 + \epsilon$.


In [ ]:
np.random.seed(42)
n_samples = 30

def f_true(x):
    return 12.0 + 3.2 * x - 0.22 * (x ** 2)

X = np.sort(np.random.uniform(1.0, 14.0, n_samples))
noise_sigma = 2.0
y = f_true(X) + np.random.normal(0, noise_sigma, n_samples)

X_eval = np.linspace(0.5, 14.5, 200)
y_eval_true = f_true(X_eval)

degrees = [1, 2, 14]
titles = ['Underfitting (Derajat 1 - Garis Lurus)', 'Optimal (Derajat 2 - Kuadratik)', 'Overfitting (Derajat 14 - Osilasi Liar)']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

models = {}
for idx, d in enumerate(degrees):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    pipe.fit(X[:, np.newaxis], y)
    models[d] = pipe
    y_pred_eval = pipe.predict(X_eval[:, np.newaxis])
    
    # Hitung MSE Latih
    mse_tr = mean_squared_error(y, pipe.predict(X[:, np.newaxis]))
    
    axes[idx].scatter(X, y, color='#2c3e50', alpha=0.7, s=40, label='Data Sampel Lapangan')
    axes[idx].plot(X_eval, y_eval_true, color='#27ae60', linestyle='--', linewidth=2, label='Fungsi Sejati (f(x))')
    axes[idx].plot(X_eval, y_pred_eval, color='#c0392b' if d==14 else ('#2980b9' if d==1 else '#e67e22'), 
                   linewidth=2.5, label=f'Model Prediksi (MSE={mse_tr:.2f})')
    axes[idx].set_title(titles[idx], fontweight='bold', fontsize=10.5)
    axes[idx].set_xlabel('Dosis Nitrogen (kg/ha)', fontweight='bold')
    if idx == 0:
        axes[idx].set_ylabel('Biomassa Tanaman (Ton/Ha)', fontweight='bold')
    axes[idx].legend(loc='lower center', fontsize=8)
    axes[idx].set_ylim(5, 30)

plt.tight_layout()
plt.savefig('docs/assets/praktikum_perbandingan_polinomial.png', bbox_inches='tight')
plt.close()
print("Visualisasi perbandingan derajat polinomial berhasil digenerate.")


## 2. Pembuktian Empiris Dekomposisi Bias-Variansi Melalui Simulasi Bootstrap

Kita akan membangkitkan $B = 100$ dataset pelatihan independen dari populasi sejati yang sama, lalu mengestimasi model pada tiap dataset.
Komponen dihitung pada titik uji baru $x_0 = 7.0$:
- $\text{Bias}^2 = (\bar{f}(x_0) - f(x_0))^2$
- $\text{Variance} = \frac{1}{B} \sum_{b=1}^B (\hat{f}_b(x_0) - \bar{f}(x_0))^2$
- $\text{Expected MSE} \approx \text{Bias}^2 + \text{Variance} + \sigma^2$


In [ ]:
B = 100
x_test_point = np.array([[7.0]])
y_true_point = f_true(7.0)

uji_derajat = [1, 2, 4, 8, 12]
hasil_dekomposisi = []

for d in uji_derajat:
    preds_b = []
    for b in range(B):
        # Bangkitkan dataset latih baru
        X_b = np.random.uniform(1.0, 14.0, n_samples)
        y_b = f_true(X_b) + np.random.normal(0, noise_sigma, n_samples)
        
        pipe = Pipeline([
            ('poly', PolynomialFeatures(degree=d, include_bias=False)),
            ('scaler', StandardScaler()),
            ('reg', LinearRegression())
        ])
        pipe.fit(X_b[:, np.newaxis], y_b)
        preds_b.append(pipe.predict(x_test_point)[0])
        
    preds_b = np.array(preds_b)
    f_bar = np.mean(preds_b)
    bias_sq = (f_bar - y_true_point) ** 2
    variance = np.var(preds_b)
    irreducible = noise_sigma ** 2
    total_expected = bias_sq + variance + irreducible
    
    hasil_dekomposisi.append({
        'Derajat Polinomial': d,
        'Bias^2': bias_sq,
        'Variance': variance,
        'Irreducible Error (sigma^2)': irreducible,
        'Total Expected Error': total_expected
    })

df_bv = pd.DataFrame(hasil_dekomposisi)
print(df_bv.to_string(index=False))


## 3. Diagnostik Kurva Pembelajaran (*Learning Curves*)

Menggunakan fungsi `learning_curve` dari Scikit-Learn untuk menginspeksi konvergensi performa terhadap variasi jumlah sampel pelatihan.


In [ ]:
# Gunakan model overfit (Derajat 8) dan model optimal (Derajat 2)
train_sizes = np.linspace(0.2, 1.0, 8)

train_sz, tr_scores_opt, val_scores_opt = learning_curve(
    models[2], X[:, np.newaxis], y, train_sizes=train_sizes, cv=5, scoring='neg_mean_squared_error'
)

train_sz, tr_scores_over, val_scores_over = learning_curve(
    models[14], X[:, np.newaxis], y, train_sizes=train_sizes, cv=5, scoring='neg_mean_squared_error'
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Model Optimal
ax1.plot(train_sz, -np.mean(tr_scores_opt, axis=1), marker='o', label='Training MSE', color='#2980b9')
ax1.plot(train_sz, -np.mean(val_scores_opt, axis=1), marker='s', linestyle='--', label='Validation MSE', color='#e67e22')
ax1.set_title('Learning Curve: Model Optimal (Derajat 2)', fontweight='bold')
ax1.set_xlabel('Jumlah Sampel Pelatihan', fontweight='bold')
ax1.set_ylabel('MSE (Galat Rerata Kuadrat)', fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 15)

# Model Overfit
ax2.plot(train_sz, -np.mean(tr_scores_over, axis=1), marker='o', label='Training MSE', color='#2980b9')
ax2.plot(train_sz, -np.mean(val_scores_over, axis=1), marker='s', linestyle='--', label='Validation MSE', color='#e67e22')
ax2.set_title('Learning Curve: Model Overfitting (Derajat 14)', fontweight='bold')
ax2.set_xlabel('Jumlah Sampel Pelatihan', fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 50)

plt.tight_layout()
plt.savefig('docs/assets/praktikum_learning_curve_diagnosis.png', bbox_inches='tight')
plt.close()
print("Grafik Learning Curves berhasil dianalisis.")


## 4. Mitigasi Overfitting dengan Regularisasi Ridge dan Lasso

Kita terapkan penalti $\ell_2$ (Ridge) dan $\ell_1$ (Lasso) pada model polinomial derajat 14 untuk menekan variansi liar.


In [ ]:
# Bandingkan OLS vs Ridge vs Lasso pada Polinomial Derajat 14
alphas = [0.01, 1.0, 10.0]

pipe_ridge = Pipeline([
    ('poly', PolynomialFeatures(degree=14, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=5.0))
])
pipe_ridge.fit(X[:, np.newaxis], y)

pipe_lasso = Pipeline([
    ('poly', PolynomialFeatures(degree=14, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Lasso(alpha=0.2, max_iter=5000))
])
pipe_lasso.fit(X[:, np.newaxis], y)

y_pred_ridge = pipe_ridge.predict(X_eval[:, np.newaxis])
y_pred_lasso = pipe_lasso.predict(X_eval[:, np.newaxis])

plt.figure(figsize=(9, 4.5))
plt.scatter(X, y, color='#2c3e50', alpha=0.7, label='Sampel Latih')
plt.plot(X_eval, y_eval_true, color='#27ae60', linestyle='--', label='Fungsi Sejati f(x)')
plt.plot(X_eval, models[14].predict(X_eval[:, np.newaxis]), color='#c0392b', alpha=0.5, label='Tanpa Regularisasi (Overfit Liar)')
plt.plot(X_eval, y_pred_ridge, color='#2980b9', linewidth=2.5, label='Mitigasi Ridge (L2, alpha=5.0)')
plt.plot(X_eval, y_pred_lasso, color='#8e44ad', linewidth=2.5, linestyle='-.', label='Mitigasi Lasso (L1, alpha=0.2)')

plt.title('Mitigasi Overfitting Menggunakan Regularisasi Ridge dan Lasso', fontweight='bold')
plt.xlabel('Dosis Pupuk Nitrogen (kg/ha)', fontweight='bold')
plt.ylabel('Biomassa Tanaman (Ton/Ha)', fontweight='bold')
plt.legend()
plt.ylim(5, 30)
plt.tight_layout()
plt.savefig('docs/assets/praktikum_mitigasi_regularisasi.png', bbox_inches='tight')
plt.close()

coef_lasso = pipe_lasso.named_steps['reg'].coef_
print(f"Banyaknya Fitur Polinomial Aktif pada Lasso: {np.sum(coef_lasso != 0)} dari 14 fitur!")


## 5. Mitigasi Overfitting pada Pohon Keputusan: Cost-Complexity Pruning

Pohon keputusan tanpa batasan kedalaman akan menghafal seluruh sampel latih ($R^2 = 1.0$).
Kita menerapkan penalti biaya kompleksitas (*cost-complexity pruning* $\alpha$).


In [ ]:
tree_unpruned = DecisionTreeRegressor(random_state=42)
tree_unpruned.fit(X[:, np.newaxis], y)

path = tree_unpruned.cost_complexity_pruning_path(X[:, np.newaxis], y)
ccp_alphas = path.ccp_alphas

# Pilih alpha optimal di tengah path
alpha_opt = ccp_alphas[len(ccp_alphas) // 2]
tree_pruned = DecisionTreeRegressor(random_state=42, ccp_alpha=alpha_opt)
tree_pruned.fit(X[:, np.newaxis], y)

mse_unpruned = mean_squared_error(y, tree_unpruned.predict(X[:, np.newaxis]))
mse_pruned = mean_squared_error(y, tree_pruned.predict(X[:, np.newaxis]))

print(f"Pohon Tanpa Pruning : Jumlah Daun = {tree_unpruned.get_n_leaves()}, MSE Latih = {mse_unpruned:.4f}")
print(f"Pohon Terpangkas    : Jumlah Daun = {tree_pruned.get_n_leaves()}, MSE Latih = {mse_pruned:.4f}")


## 6. Latihan Terpandu Mandiri (Tugas Mahasiswa)

1. **Eksplorasi Hiperparameter $\alpha$ Ridge**: Gunakan `RidgeCV` dengan rentang $\alpha \in [10^{-3}, 10^3]$ untuk mencari koefisien penalti paling optimal yang meminimalkan galat validasi silang pada estimasi biomassa sawit.
2. **Pola Underfitting Ekstrem**: Ambil model regresi linier sederhana (derajat 1) lalu terapkan regularisasi Ridge dengan $\alpha = 10^5$. Amati dan jelaskan apa yang terjadi pada garis prediksi serta nilai koefisien bobot $\mathbf{w}$.
3. **Simulasi Augmentasi Sampel**: Pada model polinomial derajat 14, naikkan jumlah data latih dari $n = 30$ menjadi $n = 1.000$. Amati bagaimana kurva prediksi berangsur-angsur jinak mendekati fungsi sejati $f(x)$ tanpa perlu regularisasi tambahan.
